# Notebook 02 - Lam sach du lieu Ha Noi

Muc tieu:
- Doc du lieu tho da enrich truong sau.
- Chon pham vi linh hoat: 1 quan hoac top-N quan de du so mau train.
- Lam sach missing, loai outlier, tao tap du lieu san sang train.
- Luu ra `data/processed/housing_hanoi_clean.csv`.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "housing_raw_vn_hanoi_deep.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH

WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/data/raw/housing_raw_vn_hanoi_deep.csv')

In [3]:
df = pd.read_csv(RAW_PATH)
print("Shape ban dau:", df.shape)
df.head(3)

Shape ban dau: (2000, 28)


,source,city,title,price_raw,area_raw,location_raw,detail_url,list_id,category_name,crawl_page,...,SoTang,Loai,GiayTo,TinhTrangNoiThat,Phuong,Quan,DiaChi,moTa,latitude,longitude
0,chotot,Ha Noi,50m2 mặt tiền phố Cự Lộc - Nở Hậu,25 tỷ,50.0,"Phường Thượng Đình, Ha Noi",https://cdn.chotot.com/4FV4LID1iLFZ0XzDHrJ-Gaa...,131635409,Nhà ở,1,...,NaN,"Nhà mặt phố, mặt tiền",Đã có sổ,NaN,Phường Thượng Đình,Quận Thanh Xuân,"36, cự lộc, Phường Thượng Đình, Quận Thanh Xuâ...","🏡 CHÍNH CHỦ BÁN NHÀ PHỐ CỰ LỘC – Ô TÔ VÀO NHÀ,...",21.000605,105.815820
1,chotot,Ha Noi,🔥SIÊU PHẨM XALA - LÔ GÓC- 3 MẶT TIỀN - 3 Ô TÔ ...,15 tỷ,62.0,"Phường Hà Cầu, Ha Noi",https://cdn.chotot.com/pqhDNAU0ftjz3UE2xCd8uy_...,131754028,Nhà ở,1,...,4.0,"Nhà mặt phố, mặt tiền",Đã có sổ,Nội thất đầy đủ,Phường Hà Cầu,Quận Hà Đông,"Xala, Phường Hà Cầu, Quận Hà Đông, Hà Nội",Dự án: \nThông tin chi tiết: 🔥ĐỘC QUYỀN XA LA:...,20.963642,105.777664
2,chotot,Ha Noi,"BÁN NHÀ 7 TẦNG CẠNH HỒ THẠCH BÀN, MẶT ĐƯỜNG Ô ...","21,5 tỷ",50.0,"Phường Thạch Bàn, Ha Noi",https://cdn.chotot.com/ejnaEJlfzzb-ZWmhOeTwUjo...,131475977,Nhà ở,1,...,7.0,"Nhà mặt phố, mặt tiền",Đã có sổ,Nội thất cao cấp,Phường Thạch Bàn,Quận Long Biên,"Đường Thạch Bàn, Phường Thạch Bàn, Quận Long B...",🔥 SIÊU PHẨM CẠNH HỒ THẠCH BÀN – 7 TẦNG THANG M...,21.025959,105.909584


## 1) Chon 1 quan theo so luong mau

In [4]:
district_counts = df["Quan"].fillna("Khong ro").value_counts()
district_counts.head(15)

Quan
Quận Long Biên       259
Quận Hoàng Mai       246
Quận Hai Bà Trưng    223
Quận Đống Đa         210
Quận Thanh Xuân      160
Quận Cầu Giấy        145
Quận Hà Đông         133
Quận Bắc Từ Liêm      90
Khong ro              86
Quận Nam Từ Liêm      83
Quận Tây Hồ           77
Quận Ba Đình          75
Huyện Thanh Trì       65
Huyện Hoài Đức        43
Huyện Đông Anh        23
Name: count, dtype: int64

In [5]:
# Luon chon duy nhat 1 quan co nhieu mau nhat (bo qua gia tri khong ro)
valid_district_counts = district_counts[district_counts.index != "Khong ro"]
TARGET_DISTRICT = valid_district_counts.index[0]
selected_districts = [TARGET_DISTRICT]

df1 = df[df["Quan"].isin(selected_districts)].copy()
print("Quan duoc chon:", TARGET_DISTRICT)
print("So mau trong quan:", len(df1))

Quan duoc chon: Quận Long Biên
So mau trong quan: 259


## 2) Chuan hoa kieu du lieu

In [6]:
num_cols = [
    "Gia", "dienTich", "chieuNgang", "chieuDai",
    "Phongngu", "PhongTam", "SoTang", "latitude", "longitude"
]

for c in num_cols:
    if c in df1.columns:
        df1[c] = pd.to_numeric(df1[c], errors="coerce")

df1[num_cols].dtypes

Gia           float64
dienTich      float64
chieuNgang    float64
chieuDai      float64
Phongngu      float64
PhongTam      float64
SoTang        float64
latitude      float64
longitude     float64
dtype: object

## 3) Xu ly missing co ban

In [7]:
print("Missing truoc:\n", df1[num_cols + ["Loai", "GiayTo", "TinhTrangNoiThat"]].isna().sum())

Missing truoc:
 Gia                   0
dienTich              0
chieuNgang          103
chieuDai            140
Phongngu              0
PhongTam             76
SoTang               51
latitude              0
longitude             0
Loai                  0
GiayTo               48
TinhTrangNoiThat    101
dtype: int64


In [8]:
# Cac cot bat buoc toi thieu cho bai toan gia nha
df1 = df1.dropna(subset=["Gia", "dienTich"])

# Dien median cho cot so (giu lai nhieu mau hon)
for c in ["Phongngu", "PhongTam", "SoTang", "chieuNgang", "chieuDai"]:
    if c in df1.columns:
        df1[c] = df1[c].fillna(df1[c].median())

# Dien nhan 'Khong ro' cho cot phan loai
for c in ["Loai", "GiayTo", "TinhTrangNoiThat", "Phuong", "Quan"]:
    if c in df1.columns:
        df1[c] = df1[c].fillna("Khong ro")

print("Shape sau xu ly missing:", df1.shape)

Shape sau xu ly missing: (259, 28)


## 4) Loai outlier co ban

In [9]:
# Dieu kien hop ly so bo (noi rong de khong loai qua nhieu mau)
cond = (
    (df1["Gia"] >= 2e8) & (df1["Gia"] <= 5e11) &
    (df1["dienTich"] >= 8) & (df1["dienTich"] <= 1500) &
    (df1["Phongngu"] >= 0) & (df1["Phongngu"] <= 30) &
    (df1["PhongTam"] >= 0) & (df1["PhongTam"] <= 30) &
    (df1["SoTang"] >= 0) & (df1["SoTang"] <= 50)
)
df2 = df1[cond].copy()

df2["Gia_m2"] = df2["Gia"] / df2["dienTich"]
print("Shape sau loc outlier:", df2.shape)
df2[["Gia", "dienTich", "Gia_m2"]].describe()

Shape sau loc outlier: (258, 29)


,Gia,dienTich,Gia_m2
count,2.580000e+02,258.000000,2.580000e+02
mean,1.571809e+10,62.727905,2.491909e+08
std,1.271312e+10,42.104125,8.910815e+07
min,3.300000e+09,20.000000,4.000000e+07
25%,8.476000e+09,40.000000,1.900000e+08
50%,1.190000e+10,50.000000,2.409946e+08
75%,1.842500e+10,66.000000,2.959567e+08
max,9.000000e+10,312.000000,5.625000e+08


## 5) Chon cot dau vao train

In [11]:
final_cols = [
    "Gia", "Gia_m2", "dienTich", "chieuNgang", "chieuDai",
    "Phongngu", "PhongTam", "SoTang",
    "Loai", "GiayTo", "TinhTrangNoiThat", "Phuong", "Quan",
     "list_id"
]

final_cols = [c for c in final_cols if c in df2.columns]
df_clean = df2[final_cols].drop_duplicates(subset=["list_id"]).copy()
print("Shape cuoi:", df_clean.shape)
df_clean.head(3)

Shape cuoi: (258, 14)


,Gia,Gia_m2,dienTich,chieuNgang,chieuDai,Phongngu,PhongTam,SoTang,Loai,GiayTo,TinhTrangNoiThat,Phuong,Quan,list_id
2,2.150000e+10,4.300000e+08,50.0,4.5000,11.2500,4.0,5.0,7.0,"Nhà mặt phố, mặt tiền",Đã có sổ,Nội thất cao cấp,Phường Thạch Bàn,Quận Long Biên,131475977
5,1.080000e+10,1.588235e+08,68.0,4.5000,11.2500,4.0,4.0,4.0,"Nhà ngõ, hẻm",Khong ro,Khong ro,Phường Thạch Bàn,Quận Long Biên,131756849
13,7.200000e+09,1.600000e+08,45.0,4.6999,9.6998,3.0,4.0,5.0,"Nhà mặt phố, mặt tiền",Đã có sổ,Nội thất đầy đủ,Phường Phúc Lợi,Quận Long Biên,132019978


In [13]:
out_path = PROCESSED_DIR / "housing_hanoi_clean.csv"
df_clean.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Da luu du lieu clean tai: {out_path}")

Da luu du lieu clean tai: C:\Users\asus\Desktop\DuDoanGiaNha\data\processed\housing_hanoi_clean.csv
